# Compound Library Standardization Pipeline

This notebook implements a standardized, logically ordered workflow for preparing a raw compound library for virtual screening. Each step is positioned to maximise computational efficiency (cheap filters first) and chemical correctness (deduplication only after canonical SMILES are established).

## Pipeline overview

| Step | Operation | Rationale |
|------|-----------|-----------|
| 0 | Environment setup | – |
| 1 | Assembly & column selection | merge multi-source CSVs |
| 2 | SMILES validity check | discard unparseable entries early |
| 3 | Desalting + largest fragment | remove counterions before any comparison |
| 4 | Canonical SMILES | enforce unambiguous representation |
| 5 | Deduplication | one correct dedup, on canonical SMILES |
| 6 | Physicochemical filters | cheap, high-throughput, large attrition |
| 7 | SA score filter | synthetic accessibility, more expensive |
| 8 | PAINS / structural alert filter | remove assay interfering compounds |
| 9 | Murcko scaffold clustering | reduce chemotype-level redundancy |
| 10 | MaxMin diversity selection | fine-grained fingerprint-space diversity |

> **Paths:** Replace all `INPUT_*` / `OUTPUT_*` variables in each cell's configuration block with your actual file paths before running.


## Step 0 – Environment Setup

All dependencies are part of a standard RDKit installation, with the exception of `sascorer.py` (RDKit Contrib, [GitHub](https://github.com/rdkit/rdkit/tree/master/Contrib/SA_Score)). Place `sascorer.py` in the directory specified by `SA_SCORER_DIR` below.


In [ ]:
import os
import sys
import time
import random
import numpy as np
import pandas as pd
import glob
from joblib import Parallel, delayed

from rdkit import Chem, DataStructs
from rdkit.Chem import Descriptors, rdMolDescriptors, rdFingerprintGenerator
from rdkit.Chem.SaltRemover import SaltRemover
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem.FilterCatalog import FilterCatalog, FilterCatalogParams
from rdkit.SimDivFilters import rdSimDivPickers

# ── SA scorer (RDKit Contrib) ──────────────────────────────────────────────────
SA_SCORER_DIR = r"path/to/sa_score"   # folder containing sascorer.py
if SA_SCORER_DIR not in sys.path:
    sys.path.append(SA_SCORER_DIR)

try:
    import sascorer
    print("sascorer loaded OK.")
except ImportError:
    raise ImportError(
        "sascorer.py not found. Download it from:\n"
        "https://github.com/rdkit/rdkit/blob/master/Contrib/SA_Score/sascorer.py\n"
        "and place it in SA_SCORER_DIR."
    )

# ── Global parallelisation setting ────────────────────────────────────────────
NUM_CORES = 4   # adjust to your machine; -1 = all available cores

print(f"RDKit and all imports OK. Using {NUM_CORES} cores for parallel steps.")


## Step 1 – Library Assembly & Column Selection

Multiple CSV files from different sources (e.g. CHEESE similarity search results across vendor databases) are merged into a single DataFrame. Only the columns required for downstream processing are retained; all others are dropped immediately to keep memory footprint low.


In [ ]:
# ── CONFIGURATION ──────────────────────────────────────────────────────────────
LIBRARY_DIR     = r"path/to/raw_library_csvs"
COLUMNS_TO_KEEP = ["smiles", "id", "database", "db_id", "similarity"]
OUTPUT_STEP1    = "01_library_assembled.csv"
# ──────────────────────────────────────────────────────────────────────────────

all_files = glob.glob(os.path.join(LIBRARY_DIR, "*.csv"))
if not all_files:
    raise FileNotFoundError(f"No CSV files found in: {LIBRARY_DIR}")

print(f"Found {len(all_files)} source CSV files.")

df_list = []
for f in all_files:
    try:
        df_list.append(pd.read_csv(f, encoding="utf-8"))
    except UnicodeDecodeError:
        df_list.append(pd.read_csv(f, encoding="latin-1"))

df = pd.concat(df_list, ignore_index=True)

# Retain only required columns; warn if any are missing
missing = [c for c in COLUMNS_TO_KEEP if c not in df.columns]
if missing:
    print(f"Warning: columns not found and skipped: {missing}")
    COLUMNS_TO_KEEP = [c for c in COLUMNS_TO_KEEP if c in df.columns]

df = df[COLUMNS_TO_KEEP].copy()
df.to_csv(OUTPUT_STEP1, index=False)

print(f"Assembled  : {len(df):,} rows from {len(all_files)} files")
print(f"Columns    : {df.columns.tolist()}")
print(f"Output     : {OUTPUT_STEP1}")


## Step 2 – SMILES Validity Check

Before any chemistry is performed, each SMILES string is passed through RDKit's parser. Entries that cannot be converted to a valid `Mol` object are discarded. This is the cheapest possible filter and should always come first.


In [ ]:
# ── CONFIGURATION ──────────────────────────────────────────────────────────────
OUTPUT_STEP2 = "02_library_valid.csv"
# ──────────────────────────────────────────────────────────────────────────────

def is_valid_smiles(smiles: str) -> bool:
    if not isinstance(smiles, str) or not smiles.strip():
        return False
    return Chem.MolFromSmiles(smiles) is not None

n_before = len(df)
valid_mask = df["smiles"].astype(str).apply(is_valid_smiles)
df = df[valid_mask].copy().reset_index(drop=True)

df.to_csv(OUTPUT_STEP2, index=False)

print(f"Input          : {n_before:,}")
print(f"Valid SMILES   : {len(df):,}")
print(f"Discarded      : {n_before - len(df):,}  (unparseable SMILES)")
print(f"Output         : {OUTPUT_STEP2}")


## Step 3 – Desalting & Largest Fragment Selection

Counterions and salt fragments (e.g. `.Cl`, `.Na`, `.TFA`) are stripped using RDKit's `SaltRemover`. When multiple disconnected fragments remain after desalting, the largest one (by heavy atom count) is retained. 

Desalting **must** precede canonical SMILES generation and deduplication: `CC(=O)[O-].[Na+]` and `CC(=O)O` would otherwise survive as two distinct entries, only to be correctly merged in a later step.


In [ ]:
# ── CONFIGURATION ──────────────────────────────────────────────────────────────
OUTPUT_STEP3 = "03_library_desalted.csv"
# ──────────────────────────────────────────────────────────────────────────────

remover = SaltRemover()

def desalt(smiles: str):
    """
    Strips salts and returns the SMILES of the largest remaining fragment.
    Returns None if the molecule cannot be processed.
    """
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None

        mol_clean = remover.StripMol(mol, dontRemoveEverything=True)

        # If multiple fragments remain, keep the largest by heavy atom count
        frags = Chem.GetMolFrags(mol_clean, asMols=True)
        if not frags:
            return None
        largest = max(frags, key=lambda m: m.GetNumHeavyAtoms())

        return Chem.MolToSmiles(largest)
    except Exception:
        return None

print(f"Desalting {len(df):,} molecules with {NUM_CORES} workers…")
t0 = time.time()

desalted = Parallel(n_jobs=NUM_CORES, verbose=5)(
    delayed(desalt)(s) for s in df["smiles"].astype(str)
)

df = df.copy()
df["smiles_desalted"] = desalted

n_before = len(df)
df = df.dropna(subset=["smiles_desalted"]).copy()
df.drop(columns=["smiles"], inplace=True)
df.rename(columns={"smiles_desalted": "smiles"}, inplace=True)

df.to_csv(OUTPUT_STEP3, index=False)

print(f"Elapsed        : {time.time()-t0:.1f} s")
print(f"Input          : {n_before:,}")
print(f"After desalt   : {len(df):,}")
print(f"Discarded      : {n_before - len(df):,}")
print(f"Output         : {OUTPUT_STEP3}")


## Step 4 – Canonical SMILES

Different databases and drawing tools produce different SMILES strings for the same molecule (varying atom ordering, aromaticity perception, stereochemistry notation). Converting all structures to RDKit canonical SMILES gives every molecule a single, deterministic string representation — a prerequisite for reliable deduplication in Step 5.


In [ ]:
# ── CONFIGURATION ──────────────────────────────────────────────────────────────
OUTPUT_STEP4 = "04_library_canonical.csv"
# ──────────────────────────────────────────────────────────────────────────────

def canonicalize(smiles: str):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        return Chem.MolToSmiles(mol, canonical=True)
    except Exception:
        return None

print(f"Canonicalizing {len(df):,} molecules with {NUM_CORES} workers…")
t0 = time.time()

canonical = Parallel(n_jobs=NUM_CORES, verbose=5)(
    delayed(canonicalize)(s) for s in df["smiles"].astype(str)
)

df = df.copy()
df["smiles_canonized"] = canonical
df = df.dropna(subset=["smiles_canonized"]).reset_index(drop=True)

df.to_csv(OUTPUT_STEP4, index=False)

print(f"Elapsed        : {time.time()-t0:.1f} s")
print(f"Canonized      : {len(df):,}")
print(f"Output         : {OUTPUT_STEP4}")


## Step 5 – Deduplication

With canonical SMILES established, exact structural duplicates are removed in a single, definitive step. This correctly collapses:
- identical structures from different source databases
- the same molecule written with different SMILES conventions (caught by canonicalization)
- salt/free-acid pairs (caught by desalting)

The first occurrence is retained; where a `similarity` column is present, the DataFrame should be sorted descending by similarity before this step so the highest-confidence entry survives.


In [ ]:
# ── CONFIGURATION ──────────────────────────────────────────────────────────────
OUTPUT_STEP5 = "05_library_dedup.csv"
# ──────────────────────────────────────────────────────────────────────────────

# Sort by similarity (descending) so the best hit per structure is kept
if "similarity" in df.columns:
    df = df.sort_values("similarity", ascending=False)

n_before = len(df)
df = df.drop_duplicates(subset=["smiles_canonized"], keep="first").copy()
df = df.reset_index(drop=True)

df.to_csv(OUTPUT_STEP5, index=False)

print(f"Before dedup   : {n_before:,}")
print(f"After dedup    : {len(df):,}")
print(f"Removed        : {n_before - len(df):,}  structural duplicates")
print(f"Output         : {OUTPUT_STEP5}")


## Step 6 – Physicochemical Property Filters

Drug-like property windows are applied before the more expensive SA score and structural alert calculations. Filtering ~millions of molecules on MW, cLogP, and TPSA takes seconds and typically removes 30–60% of a raw library, substantially reducing the computational load of all subsequent steps.

Property windows are set for oral drug-like leads. Adjust the configuration block for fragment libraries, natural product collections, or other design objectives.


In [ ]:
# ── CONFIGURATION ──────────────────────────────────────────────────────────────
OUTPUT_STEP6    = "06_library_physchem.csv"

MW_MIN,   MW_MAX     = 250, 550
LOGP_MIN, LOGP_MAX   = 1.0, 5.0
TPSA_MIN, TPSA_MAX   = 40,  120
HBD_MAX              = 3
HBA_MAX              = 10
ROT_MAX              = 8
RINGS_MIN, RINGS_MAX = 2, 6
FSP3_MIN,  FSP3_MAX  = 0.05, 0.80
# ──────────────────────────────────────────────────────────────────────────────

def calc_props(smiles: str):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return {
        "MW"           : Descriptors.MolWt(mol),
        "cLogP"        : Descriptors.MolLogP(mol),
        "TPSA"         : rdMolDescriptors.CalcTPSA(mol),
        "HBD"          : rdMolDescriptors.CalcNumHBD(mol),
        "HBA"          : rdMolDescriptors.CalcNumHBA(mol),
        "RotBonds"     : rdMolDescriptors.CalcNumRotatableBonds(mol),
        "Rings"        : rdMolDescriptors.CalcNumRings(mol),
        "AromRings"    : rdMolDescriptors.CalcNumAromaticRings(mol),
        "FractionCSP3" : rdMolDescriptors.CalcFractionCSP3(mol),
    }

print(f"Computing physicochemical properties for {len(df):,} molecules…")
t0 = time.time()

props_list = Parallel(n_jobs=NUM_CORES, verbose=5)(
    delayed(calc_props)(smi) for smi in df["smiles_canonized"].astype(str)
)

props_df = pd.DataFrame(props_list)
df = pd.concat([df.reset_index(drop=True), props_df], axis=1)

# Apply filters
mask = (
    df["MW"].between(MW_MIN, MW_MAX) &
    df["cLogP"].between(LOGP_MIN, LOGP_MAX) &
    df["TPSA"].between(TPSA_MIN, TPSA_MAX) &
    (df["HBD"] <= HBD_MAX) &
    (df["HBA"] <= HBA_MAX) &
    (df["RotBonds"] <= ROT_MAX) &
    df["Rings"].between(RINGS_MIN, RINGS_MAX) &
    df["FractionCSP3"].between(FSP3_MIN, FSP3_MAX)
)

n_before = len(df)
df = df[mask].copy().reset_index(drop=True)

df.to_csv(OUTPUT_STEP6, index=False)

print(f"Elapsed        : {time.time()-t0:.1f} s")
print(f"Input          : {n_before:,}")
print(f"After filters  : {len(df):,}")
print(f"Removed        : {n_before - len(df):,}  ({(n_before-len(df))/n_before*100:.1f}%)")
print(f"Output         : {OUTPUT_STEP6}")


## Step 7 – Synthetic Accessibility Filter

SA score (Ertl & Schuffenhauer, 2009) estimates synthetic accessibility on a scale from 1 (trivially synthesisable) to 10 (practically impossible). The cutoff of 4.0 is a pragmatic threshold for commercial library screening: molecules above this value are generally considered difficult to synthesise or resupply.

SA score is placed **after** physicochemical filters because it is computationally more expensive per molecule. Filtering the already-reduced pool from Step 6 keeps runtime proportional.


In [ ]:
# ── CONFIGURATION ──────────────────────────────────────────────────────────────
OUTPUT_STEP7 = "07_library_SA_filtered.csv"
SA_CUTOFF    = 4.0
CHUNK_SIZE   = 50_000   # process in chunks to monitor progress on large libraries
# ──────────────────────────────────────────────────────────────────────────────

def calc_sa(smiles: str):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    try:
        return sascorer.calculateScore(mol)
    except Exception:
        return None

print(f"Computing SA scores for {len(df):,} molecules with {NUM_CORES} workers…")
t0 = time.time()

sa_values = []
for start in range(0, len(df), CHUNK_SIZE):
    chunk_smiles = df["smiles_canonized"].iloc[start:start+CHUNK_SIZE].astype(str).tolist()
    chunk_sa = Parallel(n_jobs=NUM_CORES, batch_size=1000)(
        delayed(calc_sa)(smi) for smi in chunk_smiles
    )
    sa_values.extend(chunk_sa)
    print(f"  Processed {min(start+CHUNK_SIZE, len(df)):,} / {len(df):,}")

df = df.copy()
df["SA_score"] = sa_values

n_before = len(df)
df = df[df["SA_score"].notna() & (df["SA_score"] < SA_CUTOFF)].copy().reset_index(drop=True)

df.to_csv(OUTPUT_STEP7, index=False)

print(f"Elapsed        : {time.time()-t0:.1f} s")
print(f"Input          : {n_before:,}")
print(f"After SA < {SA_CUTOFF} : {len(df):,}")
print(f"Removed        : {n_before - len(df):,}")
print(f"SA score range : {df['SA_score'].min():.2f} – {df['SA_score'].max():.2f}")
print(f"Output         : {OUTPUT_STEP7}")


## Step 8 – PAINS / Structural Alert Filter

Pan-assay interference compounds (PAINS) are molecules that produce false-positive signals in biochemical assays through non-specific mechanisms: covalent reactivity, redox cycling, metal chelation, fluorescence, or colloidal aggregation. Including them in a virtual screening library wastes docking capacity and downstream experimental effort.

RDKit's `FilterCatalog` ships the PAINS-A/B/C rule sets from Baell & Holloway (2010) as a built-in catalogue — no external dependency required.


In [ ]:
# ── CONFIGURATION ──────────────────────────────────────────────────────────────
OUTPUT_STEP8 = "08_library_pains_filtered.csv"
# ──────────────────────────────────────────────────────────────────────────────

# Initialise PAINS catalogue (A + B + C rule sets)
params = FilterCatalogParams()
params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS)
pains_catalog = FilterCatalog(params)

def has_pains(smiles: str) -> bool:
    """Returns True if the molecule matches any PAINS alert."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return True   # conservative: discard unparseable entries
    return pains_catalog.HasMatch(mol)

print(f"Applying PAINS filter to {len(df):,} molecules…")
t0 = time.time()

pains_flags = Parallel(n_jobs=NUM_CORES, verbose=5)(
    delayed(has_pains)(smi) for smi in df["smiles_canonized"].astype(str)
)

df = df.copy()
df["is_pains"] = pains_flags

n_before = len(df)
df = df[~df["is_pains"]].drop(columns=["is_pains"]).reset_index(drop=True)

df.to_csv(OUTPUT_STEP8, index=False)

print(f"Elapsed        : {time.time()-t0:.1f} s")
print(f"Input          : {n_before:,}")
print(f"PAINS removed  : {n_before - len(df):,}")
print(f"Remaining      : {len(df):,}")
print(f"Output         : {OUTPUT_STEP8}")


## Step 9 – Murcko Scaffold Clustering

Generic Murcko scaffolds (ring systems with all substituents stripped and all bonds converted to single bonds) are computed for each molecule. One representative per unique scaffold is retained.

This operates at the **chemotype level**: two molecules with Tanimoto similarity of 0.85 but different ring systems will survive as separate representatives. This is intentional — scaffold diversity is a different and complementary objective to fingerprint diversity (addressed in Step 10).

Acyclic molecules (no scaffold) are retained as-is, since they represent genuinely distinct chemotypes.


In [ ]:
# ── CONFIGURATION ──────────────────────────────────────────────────────────────
OUTPUT_STEP9 = "09_library_murcko.csv"
SMILES_COL   = "smiles_canonized"
# ──────────────────────────────────────────────────────────────────────────────

def get_murcko_scaffold(smi: str):
    """
    Returns generic Murcko scaffold SMILES.
    Returns the molecule's own canonical SMILES for acyclic compounds
    (no rings → no scaffold → treat as unique).
    """
    try:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            return None
        scaff = MurckoScaffold.GetScaffoldForMol(mol)
        if scaff is None or scaff.GetNumAtoms() == 0:
            return smi   # acyclic: use molecule itself as its own "scaffold"
        return Chem.MolToSmiles(scaff, canonical=True)
    except Exception:
        return None

print(f"Computing Murcko scaffolds for {len(df):,} molecules with {NUM_CORES} workers…")
t0 = time.time()

scaffolds = Parallel(n_jobs=NUM_CORES, verbose=5)(
    delayed(get_murcko_scaffold)(smi) for smi in df[SMILES_COL].astype(str)
)

df = df.copy()
df["murcko_scaffold"] = scaffolds

n_before = len(df)
df = df.dropna(subset=["murcko_scaffold"])

# Keep one representative per scaffold (first occurrence = highest similarity hit,
# because we sorted by similarity in Step 5)
df = df.drop_duplicates(subset=["murcko_scaffold"], keep="first").copy()
df.drop(columns=["murcko_scaffold"], inplace=True)
df = df.reset_index(drop=True)

df.to_csv(OUTPUT_STEP9, index=False)

print(f"Elapsed              : {time.time()-t0:.1f} s")
print(f"Input                : {n_before:,}")
print(f"Unique scaffolds kept: {len(df):,}")
print(f"Scaffold duplicates  : {n_before - len(df):,}")
print(f"Output               : {OUTPUT_STEP9}")


## Step 10 – MaxMin Diversity Selection

The Murcko-filtered pool still contains many scaffold representatives that are fingerprint-similar (same ring system, closely related substituents). A two-pass strategy selects the final diverse panel:

**Pass A – Greedy Tanimoto clustering** (Morgan FP, radius 2, 2048 bits, cutoff = 0.70): molecules are assigned to clusters in randomised order; those that fall below the similarity threshold to all existing cluster centroids start a new cluster. One centroid per cluster is retained.

**Pass B – Property-biased MaxMin selection**: cluster centroids are ranked by their Euclidean distance from the physicochemical property centroid (z-score space). This prioritises "typical" drug-like candidates. A MaxMin similarity cap (Tanimoto ≤ 0.85) is then applied during sequential selection to enforce fingerprint diversity in the final panel.

The two passes are **complementary**: Pass A works at chemotype resolution; Pass B works at analog resolution within the same chemotype family.


In [ ]:
# ── CONFIGURATION ──────────────────────────────────────────────────────────────
OUTPUT_STEP10A = "10a_cluster_representatives.csv"
OUTPUT_STEP10B = "10b_final_panel.csv"

FP_RADIUS      = 2
FP_NBITS       = 2048
CLUSTER_CUTOFF = 0.70   # Pass A: Tanimoto threshold — below this → new cluster
MAX_SIM        = 0.85   # Pass B: max allowed Tanimoto to already-selected molecules
N_PICK         = 3000   # target panel size
RANDOM_SEED    = 42

PROP_COLS = ["MW", "cLogP", "TPSA", "RotBonds", "Rings", "FractionCSP3"]
# ──────────────────────────────────────────────────────────────────────────────

fp_gen = rdFingerprintGenerator.GetMorganGenerator(radius=FP_RADIUS, fpSize=FP_NBITS)

def get_fp(smiles: str):
    mol = Chem.MolFromSmiles(smiles)
    return fp_gen.GetFingerprint(mol) if mol else None

# ── Pass A: Greedy Tanimoto clustering ────────────────────────────────────────

df_in = df[df[SMILES_COL].notna()].copy().reset_index(drop=True)
n = len(df_in)

print(f"Computing fingerprints for {n:,} molecules…")
fps = [get_fp(smi) for smi in df_in[SMILES_COL].astype(str)]

# Randomise order to reduce order-dependence of greedy clustering
idxs = list(range(n))
random.Random(RANDOM_SEED).shuffle(idxs)

print("Pass A: greedy Tanimoto clustering…")
t0 = time.time()

centroids    = []   # indices of cluster centroid molecules
cluster_ids  = [-1] * n

for count, i in enumerate(idxs, 1):
    if fps[i] is None:
        continue
    if not centroids:
        centroids.append(i)
        cluster_ids[i] = 0
    else:
        sims     = DataStructs.BulkTanimotoSimilarity(fps[i], [fps[c] for c in centroids])
        best_sim = max(sims)
        if best_sim >= CLUSTER_CUTOFF:
            cluster_ids[i] = sims.index(best_sim)
        else:
            cluster_ids[i] = len(centroids)
            centroids.append(i)
    if count % 50_000 == 0:
        print(f"  {count:,} processed, {len(centroids):,} clusters so far")

df_reps = df_in.iloc[centroids].copy().reset_index(drop=True)
df_reps.to_csv(OUTPUT_STEP10A, index=False)

print(f"Pass A elapsed       : {time.time()-t0:.1f} s")
print(f"Cluster representatives: {len(df_reps):,}")

# ── Pass B: Property-biased MaxMin selection ───────────────────────────────────

print(f"\nPass B: property-biased MaxMin selection (target: {N_PICK:,} compounds)…")
t0 = time.time()

# Rank by distance from property-space centroid (ascending = most typical first)
prop_vals = df_reps[PROP_COLS].astype(float).copy()
for col in PROP_COLS:
    prop_vals[col].fillna(prop_vals[col].mean(), inplace=True)

z = (prop_vals - prop_vals.mean()) / prop_vals.std(ddof=0)
df_reps["prop_centrality"] = (z ** 2).sum(axis=1)
df_sorted = df_reps.sort_values("prop_centrality").reset_index(drop=True)

fps_sorted   = [get_fp(smi) for smi in df_sorted[SMILES_COL].astype(str)]
picked_idx   = []
picked_fps   = []

for i, fp in enumerate(fps_sorted):
    if fp is None:
        continue
    if len(picked_idx) >= N_PICK:
        break
    if not picked_fps or max(DataStructs.BulkTanimotoSimilarity(fp, picked_fps)) <= MAX_SIM:
        picked_idx.append(i)
        picked_fps.append(fp)
    if len(picked_idx) % 500 == 0 and len(picked_idx) > 0:
        print(f"  Selected {len(picked_idx):,}…")

df_panel = df_sorted.iloc[picked_idx].drop(columns=["prop_centrality"]).reset_index(drop=True)
df_panel.to_csv(OUTPUT_STEP10B, index=False)

print(f"Pass B elapsed       : {time.time()-t0:.1f} s")
print(f"Final panel size     : {len(df_panel):,} compounds")
print(f"Output               : {OUTPUT_STEP10B}")


## Pipeline Summary

```
Raw library
    │  Step 1   Assembly & column selection
    │  Step 2   SMILES validity check              [discard unparseable]
    │  Step 3   Desalting + largest fragment        [correct chemistry before comparison]
    │  Step 4   Canonical SMILES                   [unambiguous representation]
    │  Step 5   Deduplication                      [one correct dedup, on canonical SMILES]
    │  Step 6   Physicochemical filters             [cheap, ~seconds, large attrition]
    │  Step 7   SA score filter (< 4.0)            [synthesisability]
    │  Step 8   PAINS / structural alerts           [assay interference]
    │  Step 9   Murcko scaffold clustering          [chemotype-level redundancy]
    │  Step 10  MaxMin diversity selection          [fingerprint-level diversity]
    ▼
Final diverse, drug-like candidate panel
    → ready for high-throughput virtual screening
```

### Design principles

- **Cheap filters first**: Steps 2, 5, 6 are essentially free. Steps 7–8 are more expensive per molecule; running them on an already-reduced pool saves significant wall-clock time.
- **Deduplication once, correctly**: exact dedup on canonical SMILES (Step 5) subsumes all earlier string-level duplication. There is no need for a pre-dedup step.
- **Scaffold and fingerprint diversity are complementary**: Murcko clustering (Step 9) operates at chemotype granularity; MaxMin (Step 10) operates at analog granularity within chemotypes. Both are needed for a truly diverse panel.
